In [ ]:
from rustworkx.visualization import mpl_draw as draw_graph
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.simplefilter("ignore", UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
import pickle
from qiskit_algorithms import NumPyMinimumEigensolver
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import QAOAAnsatz

import sys
from spiq.qaoa import QAOASolver, evaluate_energy

import numpy as np
from biomarker_data import pcbo_utils
from biomarker_data.paths import sample_data_dir
from qiskit.quantum_info import SparsePauliOp


In [ ]:
n_qubits = 8
feature_set, feature_to_idx, first_corr_arr, second_corr_arr, third_corr_arr = pcbo_utils.load_features_and_corr_files(str(sample_data_dir(n_qubits)))

pcbo_obj = pcbo_utils.create_three_body_cubo(
    feature_set,
    first_corr_arr,
    second_corr_arr,
    third_corr_arr,
    feature_to_idx,
    select_n_features=4,
)

pubo = {key: float(value) for key, value in pcbo_obj.to_pubo().items()}

In [ ]:
def convert_pubo_to_ising(hypergraph: dict) -> list[tuple[str, float]]:
    """Convert a hypergraph dictionary to a list of Pauli strings with weights."""
    n = n_qubits # Number of qubits
    pauli_list = []

    for edge, weight in hypergraph.items():
        if edge:  # Ensure the edge is not empty
            # Create a Pauli string with "I" for all qubits
            paulis = ["I"] * n
            # Replace "I" with "Z" for qubits in the edge
            for node in edge:
                paulis[node] = "Z"
            # Append the reversed Pauli string and weight to the list
            pauli_list.append(("".join(paulis[::-1]), weight))

    return pauli_list

max_cut_paulis = convert_pubo_to_ising(pubo)
cost_hamiltonian = SparsePauliOp.from_list(max_cut_paulis)
paulis,coeffs = cost_hamiltonian.paulis.to_labels(),cost_hamiltonian.coeffs.real
cost_hamiltonian

In [ ]:
pcbo_obj

In [ ]:
reps=1
circuit = QAOAAnsatz(cost_operator=cost_hamiltonian, reps=reps)
# circuit.decompose(reps=2).draw()

In [ ]:
biomarker_qaoa = QAOASolver(cost_hamiltonian,circuit,"CPU")
biomarker_qaoa.prepare_circuit()
biomarker_qaoa.err = None
biomarker_qaoa.pcirc.num_parameters

In [ ]:
# Solve with classical Eigensolver for comparison
exact_solution = biomarker_qaoa.evaluate_exact_energy()

In [ ]:
# #Importing from seperate file 
# with open(f"biomarker_pickle_data/{n_qubits}_qb_biomarker_spiq_results.pkl", "rb") as f:
#     spiq_data = pickle.load(f)
# best_spiq_fitness_values = spiq_data["best_spiq_fitness_values"]
# best_spiq_parameters = spiq_data["best_spiq_parameters"]
# print("spiq_data keys:", list(spiq_data.keys()))
# biomarker_qaoa.energy_best = spiq_data['SPIQ_initialization_energy']
# biomarker_qaoa.ks_best = best_spiq_parameters[0]

In [ ]:
# we can perform SPIQ by using the main optimization function "claptonize"
biomarker_qaoa.run_spiq(n_gens=1)

In [ ]:
biomarker_qaoa.energy_best

In [ ]:
spiq_angles = [param * np.pi/2 for param in biomarker_qaoa.ks_best]

In [ ]:
energies = [biomarker_qaoa.evaluate_energy(biomarker_qaoa.pcirc,biomarker_qaoa.cost_hamiltonian,spiq_angles) for _ in range(10)]
average_energy = np.mean(energies)
print(f"Average SPIQ Qiskit Energy: {average_energy}")

In [ ]:
# Random Initalization
random_angles = np.random.random(len(biomarker_qaoa.ks_best))
random_energies = [evaluate_energy(biomarker_qaoa.pcirc, cost_hamiltonian, random_angles) for _ in range(10)]
min_energy = min(random_energies)
print(f"Minimum Energy found with Random initialization over 100 runs: {min_energy}")

In [ ]:
spiq_result,spiq_iteration_vals = biomarker_qaoa.run_qaoa(spiq_angles,max_iters=1000)
random_result,random_iteration_vals = biomarker_qaoa.run_qaoa(random_angles,max_iters=1000)

In [ ]:
spiq_result

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(spiq_iteration_vals, label="SPIQ")
plt.plot(random_iteration_vals, label="Random Initialization")
plt.legend()
plt.xlabel("Iteration")
plt.ylabel("Cost")
plt.show()